In [3]:
import os
import json
import csv
import glob

# --- CONFIGURATION DES CHEMINS ---
# Correction pour supporter Jupyter/Notebooks et l'exécution standard
try:
    # Essaie de récupérer le dossier du script
    ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Si ça échoue (ex: dans Jupyter), utilise le dossier de travail actuel
    ROOT_DIR = os.getcwd()

# Dossier contenant les dossiers par thème (animaux, sport...) avec images/sons
MEDIA_DIR = os.path.join(ROOT_DIR, "questions")
# Dossier contenant tes fichiers Excel exportés en CSV
CSV_DIR = os.path.join(ROOT_DIR, "CSV _ Questions")
# Fichier de sortie
OUTPUT_FILE = os.path.join(ROOT_DIR, "database.js")

database = {}

print("------------------------------------------------")
print(f"   GÉNÉRATEUR DE BASE DE DONNÉES DU JEU   ")
print(f"   Dossier racine détecté : {ROOT_DIR}")
print("------------------------------------------------")

# 1. INITIALISATION DE LA STRUCTURE
def get_theme_structure():
    return { 
        "1": {"images":[], "sounds":[], "text":[]},
        "2": {"images":[], "sounds":[], "text":[]},
        "3": {"images":[], "sounds":[], "text":[]},
        "4": {"images":[], "sounds":[], "text":[]},
        "5": {"images":[], "sounds":[], "text":[]}
    }

# 2. SCAN DES MÉDIAS (Dossiers questions/theme/level/...)
if os.path.exists(MEDIA_DIR):
    print(f"Scan du dossier média : {MEDIA_DIR}")
    for theme in os.listdir(MEDIA_DIR):
        theme_path = os.path.join(MEDIA_DIR, theme)
        
        if os.path.isdir(theme_path):
            # Normalisation du nom du thème (minuscule)
            theme_key = theme.lower()
            if theme_key not in database:
                database[theme_key] = get_theme_structure()
            
            # Scan des niveaux 1 à 5
            for level in range(1, 6):
                str_level = str(level)
                level_path = os.path.join(theme_path, str_level)
                
                if os.path.exists(level_path):
                    # --- IMAGES ---
                    img_path = os.path.join(level_path, "images")
                    if os.path.exists(img_path):
                        for f in os.listdir(img_path):
                            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.gif')):
                                # Chemin relatif (avec des slashes normaux pour le web)
                                full_path = f"questions/{theme}/{level}/images/{f}".replace("\\", "/")
                                database[theme_key][str_level]["images"].append(full_path)
                    
                    # --- SONS ---
                    snd_path = os.path.join(level_path, "sons")
                    if os.path.exists(snd_path):
                        for f in os.listdir(snd_path):
                            if f.lower().endswith(('.mp3', '.wav', '.ogg')):
                                full_path = f"questions/{theme}/{level}/sons/{f}".replace("\\", "/")
                                database[theme_key][str_level]["sounds"].append(full_path)

# 3. SCAN DES CSV (Questions Textuelles)
if os.path.exists(CSV_DIR):
    print(f"Scan du dossier CSV : {CSV_DIR}")
    for csv_file in glob.glob(os.path.join(CSV_DIR, "*.csv")):
        filename = os.path.basename(csv_file)
        
        # Nettoyage du nom : on enlève "questions_" et l'extension
        clean_name = filename.lower().replace("questions_", "").replace(".csv", "").strip()
        
        if clean_name not in database:
            database[clean_name] = get_theme_structure()
            
        print(f" -> Lecture de {filename} (Thème : {clean_name})")
        
        try:
            # Utf-8-sig gère le BOM d'Excel, errors='replace' évite les crashs sur caractères bizarres
            with open(csv_file, mode='r', encoding='utf-8-sig', errors='replace') as f:
                # Lecture brute pour détecter le séparateur
                sample = f.read(1024)
                f.seek(0)
                try:
                    dialect = csv.Sniffer().sniff(sample)
                except:
                    dialect = 'excel' # Fallback
                
                reader = csv.DictReader(f, dialect=dialect)
                
                count = 0
                for row in reader:
                    # Nettoyage des clés (retirer espaces et mettre en majuscule)
                    clean_row = {k.strip().upper(): v for k, v in row.items() if k}
                    
                    question = clean_row.get('QUESTION')
                    reponse = clean_row.get('REPONSE')
                    difficulte = clean_row.get('DIFFICULTE', '1')
                    
                    if question and reponse:
                        try:
                            # On convertit la difficulté en entier pour vérifier qu'elle est valide
                            lvl_int = int(difficulte)
                            if 1 <= lvl_int <= 5:
                                lvl_str = str(lvl_int)
                                database[clean_name][lvl_str]["text"].append({
                                    "question": question,
                                    "answer": reponse
                                })
                                count += 1
                        except ValueError:
                            pass 
                print(f"    + {count} questions ajoutées.")
                
        except Exception as e:
            print(f"    ERREUR sur {filename}: {e}")

# 4. ÉCRITURE DU RÉSULTAT
print("------------------------------------------------")
print("Écriture du fichier database.js...")

js_content = f"const QUESTION_DATABASE = {json.dumps(database, indent=4, ensure_ascii=False)};"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(js_content)

print(f"TERMINE ! 'database.js' a été créé.")

------------------------------------------------
   GÉNÉRATEUR DE BASE DE DONNÉES DU JEU   
   Dossier racine détecté : c:\Users\Sid\Desktop\Le Risque Tout
------------------------------------------------
Scan du dossier média : c:\Users\Sid\Desktop\Le Risque Tout\questions
Scan du dossier CSV : c:\Users\Sid\Desktop\Le Risque Tout\CSV _ Questions
 -> Lecture de questions_animaux.csv (Thème : animaux)
    + 159 questions ajoutées.
 -> Lecture de questions_annee_2000.csv (Thème : annee_2000)
    + 0 questions ajoutées.
 -> Lecture de questions_art.csv (Thème : art)
    + 0 questions ajoutées.
 -> Lecture de questions_celibrite.csv (Thème : celibrite)
    + 0 questions ajoutées.
 -> Lecture de questions_cinéma.csv (Thème : cinéma)
    + 0 questions ajoutées.
 -> Lecture de questions_cuisines.csv (Thème : cuisines)
    + 97 questions ajoutées.
 -> Lecture de questions_disney.csv (Thème : disney)
    + 103 questions ajoutées.
 -> Lecture de questions_geographie.csv (Thème : geographie)
    

In [4]:
import os
import json
import csv
import glob
import unicodedata

# --- CONFIGURATION ---
try:
    ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    ROOT_DIR = os.getcwd()

MEDIA_DIR = os.path.join(ROOT_DIR, "questions")
CSV_DIR = os.path.join(ROOT_DIR, "CSV _ Questions")
OUTPUT_FILE = os.path.join(ROOT_DIR, "database.js")

database = {}

print("------------------------------------------------")
print(f"   GÉNÉRATEUR DE BASE DE DONNÉES V2 (CORRIGÉ)   ")
print("------------------------------------------------")

def normalize_name(name):
    """ Enlève les accents et met en minuscule pour matcher les dossiers """
    if not isinstance(name, str): return str(name)
    nfkd = unicodedata.normalize('NFKD', name)
    return "".join([c for c in nfkd if not unicodedata.combining(c)]).lower().strip()

def get_theme_structure():
    return { 
        "1": {"images":[], "sounds":[], "text":[]},
        "2": {"images":[], "sounds":[], "text":[]},
        "3": {"images":[], "sounds":[], "text":[]},
        "4": {"images":[], "sounds":[], "text":[]},
        "5": {"images":[], "sounds":[], "text":[]}
    }

# 1. SCAN DES DOSSIERS (MÉDIAS)
if os.path.exists(MEDIA_DIR):
    for theme in os.listdir(MEDIA_DIR):
        theme_path = os.path.join(MEDIA_DIR, theme)
        if os.path.isdir(theme_path):
            # On utilise le nom du dossier comme clé principale
            theme_key = normalize_name(theme)
            
            if theme_key not in database:
                database[theme_key] = get_theme_structure()
            
            # Scan Niveaux
            for level in range(1, 6):
                str_level = str(level)
                level_path = os.path.join(theme_path, str_level)
                if os.path.exists(level_path):
                    # Images
                    img_path = os.path.join(level_path, "images")
                    if os.path.exists(img_path):
                        for f in os.listdir(img_path):
                            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.gif')):
                                full_path = f"questions/{theme}/{level}/images/{f}".replace("\\", "/")
                                database[theme_key][str_level]["images"].append(full_path)
                    # Sons
                    snd_path = os.path.join(level_path, "sons")
                    if os.path.exists(snd_path):
                        for f in os.listdir(snd_path):
                            if f.lower().endswith(('.mp3', '.wav', '.ogg')):
                                full_path = f"questions/{theme}/{level}/sons/{f}".replace("\\", "/")
                                database[theme_key][str_level]["sounds"].append(full_path)

# 2. SCAN DES CSV (TEXTE)
if os.path.exists(CSV_DIR):
    for csv_file in glob.glob(os.path.join(CSV_DIR, "*.csv")):
        filename = os.path.basename(csv_file)
        # Nettoyage : questions_cinéma.csv -> cinema
        raw_name = filename.lower().replace("questions_", "").replace(".csv", "").strip()
        clean_name = normalize_name(raw_name)
        
        # Mapping manuel si les noms diffèrent trop (ex: cuisines vs cuisine)
        if clean_name == "cuisines": clean_name = "cuisine"
        if clean_name == "celibrite": clean_name = "celebrites"
        if clean_name == "annee_2000": clean_name = "annees_2000"
        
        if clean_name not in database:
            database[clean_name] = get_theme_structure()
            
        print(f" -> CSV : {filename} (Mappé sur : {clean_name})")
        
        try:
            with open(csv_file, mode='r', encoding='utf-8-sig', errors='replace') as f:
                # DÉTECTION ROBUSTE DU SÉPARATEUR
                # On lit la première ligne. Si elle contient des ';', c'est sûrement ça. Sinon virgule.
                first_line = f.readline()
                delimiter = ';' if ';' in first_line else ','
                f.seek(0) # Retour au début
                
                reader = csv.DictReader(f, delimiter=delimiter)
                
                count = 0
                for row in reader:
                    # Nettoyage des clés du CSV (supprimer espaces et mettre en MAJ)
                    row_clean = {k.strip().upper(): v for k, v in row.items() if k}
                    
                    q = row_clean.get('QUESTION')
                    r = row_clean.get('REPONSE')
                    d = row_clean.get('DIFFICULTE', '1')
                    
                    if q and r:
                        try:
                            # Correction niveau (parfois '1' parfois 1)
                            lvl = str(int(float(d))) # Gère '1.0' ou '1'
                            if lvl in database[clean_name]:
                                database[clean_name][lvl]["text"].append({
                                    "question": q,
                                    "answer": r
                                })
                                count += 1
                        except:
                            pass # Erreur de niveau
                
                if count == 0:
                    print("    /!\\ ATTENTION : 0 questions trouvées. Vérifie les entêtes (QUESTION, REPONSE, DIFFICULTE)")
                else:
                    print(f"    + {count} questions ajoutées.")
                    
        except Exception as e:
            print(f"    ERREUR CRITIQUE sur {filename}: {e}")

# 3. ÉCRITURE
js_content = f"const QUESTION_DATABASE = {json.dumps(database, indent=4, ensure_ascii=False)};"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(js_content)

print("\nTERMINE ! Relance index.html pour voir les changements.")

------------------------------------------------
   GÉNÉRATEUR DE BASE DE DONNÉES V2 (CORRIGÉ)   
------------------------------------------------
 -> CSV : questions_animaux.csv (Mappé sur : animaux)
    + 159 questions ajoutées.
 -> CSV : questions_annee_2000.csv (Mappé sur : annees_2000)
    + 333 questions ajoutées.
 -> CSV : questions_art.csv (Mappé sur : art)
    + 194 questions ajoutées.
 -> CSV : questions_celibrite.csv (Mappé sur : celebrites)
    + 217 questions ajoutées.
 -> CSV : questions_cinéma.csv (Mappé sur : cinema)
    + 194 questions ajoutées.
 -> CSV : questions_cuisines.csv (Mappé sur : cuisine)
    + 97 questions ajoutées.
 -> CSV : questions_disney.csv (Mappé sur : disney)
    + 103 questions ajoutées.
 -> CSV : questions_geographie.csv (Mappé sur : geographie)
    + 294 questions ajoutées.
 -> CSV : questions_harry_potter.csv (Mappé sur : harry_potter)
    + 162 questions ajoutées.
 -> CSV : questions_histoire.csv (Mappé sur : histoire)
    + 196 questions ajout

In [2]:
import os
import json
import csv
import glob
import unicodedata
import sys

# --- 1. CONFIGURATION DES CHEMINS ---
# On détecte le dossier où se trouve le script (V9)
try:
    CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    CURRENT_DIR = os.getcwd()

# On définit les chemins basés sur ta structure V9
QUIZZ_ROOT = os.path.join(CURRENT_DIR, "Quizz")
MEDIA_DIR = os.path.join(QUIZZ_ROOT, "questions")
CSV_DIR = os.path.join(QUIZZ_ROOT, "CSV _ Questions")
OUTPUT_FILE = os.path.join(CURRENT_DIR, "database.js")

database = {}

print("------------------------------------------------")
print(f"   GÉNÉRATEUR DE BASE DE DONNÉES (V9)   ")
print(f"   Script situé dans : {CURRENT_DIR}")
print(f"   Dossier Quizz     : {QUIZZ_ROOT}")
print("------------------------------------------------")

# Vérification préliminaire
if not os.path.exists(QUIZZ_ROOT):
    print(f"/!\\ ERREUR CRITIQUE : Le dossier 'Quizz' est introuvable dans {CURRENT_DIR}.")
    print("Assure-toi que ce script est bien dans le dossier 'V9'.")
    sys.exit()

def normalize_name(name):
    """ Nettoie les noms de dossiers pour les utiliser comme ID """
    if not isinstance(name, str): return str(name)
    nfkd = unicodedata.normalize('NFKD', name)
    # On garde les caractères alphanumériques et on remplace les espaces par des underscores
    return "".join([c for c in nfkd if not unicodedata.combining(c)]).lower().strip().replace(" ", "_").replace("'", "")

def get_theme_structure():
    return { 
        "1": {"images":[], "sounds":[], "text":[]},
        "2": {"images":[], "sounds":[], "text":[]},
        "3": {"images":[], "sounds":[], "text":[]},
        "4": {"images":[], "sounds":[], "text":[]},
        "5": {"images":[], "sounds":[], "text":[]}
    }

def get_question_text(theme, type_media):
    """ Génère la question par défaut selon le thème et le média """
    if theme == 'animaux':
        return "Quel est cet animal ?" if type_media == 'images' else "À quel animal ce bruit est-il associé ?"
    elif theme in ['sport', 'celebrites', 'harry_potter', 'disney', 'cinema', 'musique']:
        return "Qui est-ce ?" if type_media == 'images' else ("Quel est ce titre ?" if theme == 'musique' else "De qui/quoi s'agit-il ?")
    elif theme == 'geographie' and type_media == 'images':
        return "Quel pays a ce drapeau ?"
    elif type_media == 'sounds':
        return "De quoi/qui s'agit-il ?"
    else:
        return "Identifiez cet élément"

def clean_filename(filename):
    """ Nettoie le nom du fichier pour en faire une réponse """
    name = os.path.splitext(filename)[0] # Enlève l'extension
    return name.replace("_", " ").replace("-", " ").title()

# --- 2. SCAN DES IMAGES ET SONS ---
if os.path.exists(MEDIA_DIR):
    print(f"Scan des médias dans : {MEDIA_DIR}")
    for theme in os.listdir(MEDIA_DIR):
        theme_path = os.path.join(MEDIA_DIR, theme)
        
        if os.path.isdir(theme_path):
            theme_key = normalize_name(theme)
            
            # Corrections manuelles pour tes dossiers spécifiques
            if theme_key == "annees_2000": theme_key = "annees_2000"
            if theme_key == "jeux_video": theme_key = "jeux_video"
            if theme_key == "harry_potter": theme_key = "harry_potter"
            if theme_key == "geographie": theme_key = "geographie"
            
            if theme_key not in database:
                database[theme_key] = get_theme_structure()
            
            # Scan des niveaux 1 à 5
            count_img = 0
            count_snd = 0
            
            for level in range(1, 6):
                str_level = str(level)
                level_path = os.path.join(theme_path, str_level)
                
                if os.path.exists(level_path):
                    # Images (récursif pour trouver 'drapeau' etc)
                    img_path = os.path.join(level_path, "images")
                    if os.path.exists(img_path):
                        for root, dirs, files in os.walk(img_path):
                            for f in files:
                                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.gif')):
                                    abs_path = os.path.join(root, f)
                                    # Chemin relatif pour le HTML : "Quizz/questions/..."
                                    rel_path = os.path.relpath(abs_path, CURRENT_DIR).replace("\\", "/")
                                    
                                    # Création de l'objet question pour l'image
                                    question_obj = {
                                        "file": rel_path,
                                        "question": get_question_text(theme_key, 'images'),
                                        "answer": clean_filename(f)
                                    }
                                    database[theme_key][str_level]["images"].append(question_obj)
                                    count_img += 1
                    
                    # Sons
                    snd_path = os.path.join(level_path, "sons")
                    if os.path.exists(snd_path):
                        for root, dirs, files in os.walk(snd_path):
                            for f in files:
                                if f.lower().endswith(('.mp3', '.wav', '.ogg')):
                                    abs_path = os.path.join(root, f)
                                    rel_path = os.path.relpath(abs_path, CURRENT_DIR).replace("\\", "/")
                                    
                                    # Création de l'objet question pour le son
                                    question_obj = {
                                        "file": rel_path,
                                        "question": get_question_text(theme_key, 'sounds'),
                                        "answer": clean_filename(f)
                                    }
                                    database[theme_key][str_level]["sounds"].append(question_obj)
                                    count_snd += 1
            
            print(f"   - {theme} ({theme_key}) : {count_img} images, {count_snd} sons.")

# --- 3. SCAN DES CSV (TEXTE) ---
if os.path.exists(CSV_DIR):
    print(f"\nScan des CSV dans : {CSV_DIR}")
    for csv_file in glob.glob(os.path.join(CSV_DIR, "*.csv")):
        filename = os.path.basename(csv_file)
        
        # Nettoyage du nom (ex: questions_animaux.csv -> animaux)
        raw_name = filename.lower().replace("questions_", "").replace(".csv", "").strip()
        clean_name = normalize_name(raw_name)
        
        # Mapping manuel (si le nom du CSV diffère du dossier)
        if clean_name == "cuisines": clean_name = "cuisine"
        if clean_name == "celibrite": clean_name = "celebrites"
        if clean_name == "cinéma": clean_name = "cinema"
        if clean_name == "cinema": clean_name = "cinema" # Cas ou l'accent est deja parti
        if clean_name == "annee_2000": clean_name = "annees_2000"
        if clean_name == "jeux_vidéo": clean_name = "jeux_video"
        if clean_name == "jeux_video": clean_name = "jeux_video" # Cas sans accent
        
        if clean_name not in database:
            database[clean_name] = get_theme_structure()
            
        try:
            with open(csv_file, mode='r', encoding='utf-8-sig', errors='replace') as f:
                # Détection séparateur
                first_line = f.readline()
                delimiter = ';' if ';' in first_line else ','
                f.seek(0)
                
                reader = csv.DictReader(f, delimiter=delimiter)
                count_txt = 0
                for row in reader:
                    row_clean = {k.strip().upper(): v for k, v in row.items() if k}
                    q = row_clean.get('QUESTION')
                    r = row_clean.get('REPONSE')
                    d = row_clean.get('DIFFICULTE', '1')
                    
                    if q and r:
                        try:
                            # Gestion des nombres à virgule (ex: 1.0 -> 1)
                            lvl = str(int(float(d)))
                            if lvl in database[clean_name]:
                                database[clean_name][lvl]["text"].append({
                                    "question": q,
                                    "answer": r
                                })
                                count_txt += 1
                        except: pass
                
                print(f"   -> {filename} ({clean_name}) : {count_txt} questions texte.")
                    
        except Exception as e:
            print(f"   /!\\ Erreur lecture {filename}: {e}")
else:
    print(f"/!\\ ERREUR : Dossier CSV introuvable : {CSV_DIR}")

# --- 4. ÉCRITURE FINALE ---
print("\n------------------------------------------------")
print("Écriture du fichier database.js...")

js_content = f"const QUESTION_DATABASE = {json.dumps(database, indent=4, ensure_ascii=False)};"

try:
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(js_content)
    print(f"SUCCÈS ! Base de données générée : {OUTPUT_FILE}")
except Exception as e:
    print(f"ERREUR D'ÉCRITURE : {e}")

print("Tu peux maintenant ouvrir index.html.")

------------------------------------------------
   GÉNÉRATEUR DE BASE DE DONNÉES (V9)   
   Script situé dans : c:\Users\Sid\Desktop\Le Risque Tout\V9
   Dossier Quizz     : c:\Users\Sid\Desktop\Le Risque Tout\V9\Quizz
------------------------------------------------
Scan des médias dans : c:\Users\Sid\Desktop\Le Risque Tout\V9\Quizz\questions
   - animaux (animaux) : 229 images, 17 sons.
   - annees_2000 (annees_2000) : 0 images, 0 sons.
   - art (art) : 0 images, 0 sons.
   - celebrites (celebrites) : 0 images, 0 sons.
   - cinema (cinema) : 0 images, 0 sons.
   - cuisine (cuisine) : 0 images, 0 sons.
   - disney (disney) : 119 images, 0 sons.
   - geographie (geographie) : 253 images, 0 sons.
   - harry_potter (harry_potter) : 95 images, 0 sons.
   - histoire (histoire) : 0 images, 0 sons.
   - jeux_video (jeux_video) : 0 images, 0 sons.
   - litterature (litterature) : 0 images, 0 sons.
   - musique (musique) : 56 images, 0 sons.
   - mystere (mystere) : 0 images, 0 sons.
   - sci